In [1]:
!pip install torch torchvision tqdm pillow numpy -q

In [2]:
import os
import random
import math
import time
import numpy as np
from PIL import Image
import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from tqdm import tqdm

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [3]:
import zipfile

BASE_PATH = "/content"

zip_files = [
    "HR_train1.zip", "HR_train2.zip", "HR_train3.zip", "HR_train4.zip",
    "LR_train1.zip", "LR_train2.zip", "LR_train3.zip", "LR_train4.zip",
    "HR_val.zip", "LR_val.zip", "HR_train.zip", "LR_train.zip"
]

for zf in zip_files:
    zip_path = os.path.join(BASE_PATH, zf)
    if os.path.exists(zip_path):
        folder_name = zf.replace(".zip", "")
        extract_dir = os.path.join(BASE_PATH, folder_name)
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_dir)
        print(f"Extracted: {zf} -> {extract_dir}")
    else:
        print(f"[SKIP] Not found: {zip_path}")

Extracted: HR_train1.zip -> /content/HR_train1
Extracted: HR_train2.zip -> /content/HR_train2
Extracted: HR_train3.zip -> /content/HR_train3
Extracted: HR_train4.zip -> /content/HR_train4
Extracted: LR_train1.zip -> /content/LR_train1
Extracted: LR_train2.zip -> /content/LR_train2
Extracted: LR_train3.zip -> /content/LR_train3
Extracted: LR_train4.zip -> /content/LR_train4
Extracted: HR_val.zip -> /content/HR_val
Extracted: LR_val.zip -> /content/LR_val
Extracted: HR_train.zip -> /content/HR_train
Extracted: LR_train.zip -> /content/LR_train


In [4]:
def collect_images(base, folders):
    """Collect all image paths from multiple folders."""
    paths = []
    for folder in folders:
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp']:
            found = glob.glob(os.path.join(base, folder, '**', ext), recursive=True)
            found += glob.glob(os.path.join(base, folder, ext))
            paths.extend(found)
    paths = sorted(list(set(paths)))
    return paths

HR_train_folders = ["HR_train1", "HR_train2", "HR_train3", "HR_train4"]
LR_train_folders = ["LR_train1", "LR_train2", "LR_train3", "LR_train4"]
HR_val_folders   = ["HR_val"]
LR_val_folders   = ["LR_val"]

hr_train_paths = collect_images(BASE_PATH, HR_train_folders)
lr_train_paths = collect_images(BASE_PATH, LR_train_folders)
hr_val_paths   = collect_images(BASE_PATH, HR_val_folders)
lr_val_paths   = collect_images(BASE_PATH, LR_val_folders)

print(f"HR Train images : {len(hr_train_paths)}")
print(f"LR Train images : {len(lr_train_paths)}")
print(f"HR Val images   : {len(hr_val_paths)}")
print(f"LR Val images   : {len(lr_val_paths)}")

HR Train images : 3009
LR Train images : 3009
HR Val images   : 100
LR Val images   : 100


In [12]:
# Verify a few pairs look correct by filename
print("Sample train pairs (first 5):")
for i in range(5):
    hr_name = os.path.basename(hr_train_paths[i])
    lr_name = os.path.basename(lr_train_paths[i])
    match = "✓" if hr_name == lr_name else "✗ MISMATCH"
    print(f"  {match}  HR: {hr_name}  |  LR: {lr_name}")

print("\nSample val pairs (first 5):")
for i in range(5):
    hr_name = os.path.basename(hr_val_paths[i])
    lr_name = os.path.basename(lr_val_paths[i])
    match = "✓" if hr_name == lr_name else "✗ MISMATCH"
    print(f"  {match}  HR: {hr_name}  |  LR: {lr_name}")

Sample train pairs (first 5):
  ✓  HR: 0000.png  |  LR: 0000.png
  ✓  HR: 0001.png  |  LR: 0001.png
  ✓  HR: 00010.png  |  LR: 00010.png
  ✓  HR: 000100.png  |  LR: 000100.png
  ✓  HR: 000101.png  |  LR: 000101.png

Sample val pairs (first 5):
  ✓  HR: 0000.png  |  LR: 0000.png
  ✓  HR: 00010.png  |  LR: 00010.png
  ✓  HR: 000100.png  |  LR: 000100.png
  ✓  HR: 000102.png  |  LR: 000102.png
  ✓  HR: 000104.png  |  LR: 000104.png


In [5]:
class SRDataset(Dataset):
    """
    Pairs LR and HR images by sorted filename.
    LR and HR images must have matching filenames across folders.
    """
    def __init__(self, lr_paths, hr_paths, augment=True, img_size=256):
        super().__init__()
        # Sort both lists by filename only (not full path) so they pair correctly
        self.lr_paths = sorted(lr_paths, key=lambda p: os.path.basename(p))
        self.hr_paths = sorted(hr_paths, key=lambda p: os.path.basename(p))
        assert len(self.lr_paths) == len(self.hr_paths), \
            f"Mismatch: {len(self.lr_paths)} LR vs {len(self.hr_paths)} HR"
        self.augment = augment
        self.img_size = img_size

    def __len__(self):
        return len(self.lr_paths)

    def __getitem__(self, idx):
        lr_img = Image.open(self.lr_paths[idx]).convert('RGB')
        hr_img = Image.open(self.hr_paths[idx]).convert('RGB')

        # Ensure both are 256x256
        lr_img = lr_img.resize((self.img_size, self.img_size), Image.BICUBIC)
        hr_img = hr_img.resize((self.img_size, self.img_size), Image.BICUBIC)

        # Data Augmentation (training only)
        if self.augment:
            # Random horizontal flip
            if random.random() > 0.5:
                lr_img = TF.hflip(lr_img)
                hr_img = TF.hflip(hr_img)
            # Random vertical flip
            if random.random() > 0.5:
                lr_img = TF.vflip(lr_img)
                hr_img = TF.vflip(hr_img)
            # Random 90-degree rotation
            if random.random() > 0.5:
                angle = random.choice([90, 180, 270])
                lr_img = TF.rotate(lr_img, angle)
                hr_img = TF.rotate(hr_img, angle)

        lr_tensor = TF.to_tensor(lr_img)   # [0, 1]
        hr_tensor = TF.to_tensor(hr_img)   # [0, 1]
        return lr_tensor, hr_tensor


In [6]:
# CELL 6: Model Architecture — MSRDAN
#   Multi-Scale Residual Dense Attention Network
#   Unique combination of:
#     - Dense connections within blocks
#     - Dual Attention (channel + spatial)
#     - Multi-scale feature extraction
#     - Global residual learning
# ============================================================

class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation style channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.fc(self.avg_pool(x))
        mx  = self.fc(self.max_pool(x))
        return x * self.sigmoid(avg + mx)


class SpatialAttention(nn.Module):
    """Spatial attention using avg + max pooling across channels."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        attn = torch.cat([avg, mx], dim=1)
        return x * self.sigmoid(self.conv(attn))


class DualAttentionBlock(nn.Module):
    """Channel + Spatial attention combined."""
    def __init__(self, channels):
        super().__init__()
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x


class DenseLayer(nn.Module):
    def __init__(self, in_ch, growth_rate):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, growth_rate, 3, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return torch.cat([x, self.conv(x)], dim=1)


class ResidualDenseBlock(nn.Module):
    """
    Dense block with 4 layers + dual attention + local residual.
    """
    def __init__(self, in_ch=64, growth_rate=32):
        super().__init__()
        self.d1 = DenseLayer(in_ch,               growth_rate)
        self.d2 = DenseLayer(in_ch + growth_rate,   growth_rate)
        self.d3 = DenseLayer(in_ch + 2*growth_rate, growth_rate)
        self.d4 = DenseLayer(in_ch + 3*growth_rate, growth_rate)
        self.compress = nn.Conv2d(in_ch + 4*growth_rate, in_ch, 1, bias=False)
        self.attn = DualAttentionBlock(in_ch)
        self.scale = nn.Parameter(torch.ones(1) * 0.2)

    def forward(self, x):
        out = self.d1(x)
        out = self.d2(out)
        out = self.d3(out)
        out = self.d4(out)
        out = self.compress(out)
        out = self.attn(out)
        return x + out * self.scale


class MultiScaleFusion(nn.Module):
    """
    Extract features at 3 scales (1x, 2x dilated, 4x dilated) and fuse them.
    """
    def __init__(self, channels):
        super().__init__()
        self.branch1 = nn.Conv2d(channels, channels, 3, padding=1,  dilation=1,  bias=False)
        self.branch2 = nn.Conv2d(channels, channels, 3, padding=2,  dilation=2,  bias=False)
        self.branch3 = nn.Conv2d(channels, channels, 3, padding=4,  dilation=4,  bias=False)
        self.fuse    = nn.Conv2d(channels * 3, channels, 1, bias=False)
        self.act     = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        b1 = self.act(self.branch1(x))
        b2 = self.act(self.branch2(x))
        b3 = self.act(self.branch3(x))
        return self.fuse(torch.cat([b1, b2, b3], dim=1))


class MSRDAN(nn.Module):
    """
    Multi-Scale Residual Dense Attention Network for Super Resolution.
    Input:  [B, 3, 256, 256]
    Output: [B, 3, 256, 256]
    """
    def __init__(self, num_channels=3, num_features=64, num_rdb=8, growth_rate=32):
        super().__init__()

        # Shallow feature extraction
        self.head = nn.Sequential(
            nn.Conv2d(num_channels, num_features, 3, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(num_features, num_features, 3, padding=1, bias=False),
        )

        # Multi-Scale feature extraction at input level
        self.ms_head = MultiScaleFusion(num_features)

        # Stack of Residual Dense Blocks
        self.rdbs = nn.ModuleList([
            ResidualDenseBlock(num_features, growth_rate)
            for _ in range(num_rdb)
        ])

        # Global feature fusion
        self.gff = nn.Sequential(
            nn.Conv2d(num_features * num_rdb, num_features, 1, bias=False),
            nn.Conv2d(num_features, num_features, 3, padding=1, bias=False),
        )

        # Global attention
        self.global_attn = DualAttentionBlock(num_features)

        # Multi-scale fusion in deep features
        self.ms_deep = MultiScaleFusion(num_features)

        # Reconstruction head
        self.tail = nn.Sequential(
            nn.Conv2d(num_features, num_features, 3, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(num_features, num_features // 2, 3, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(num_features // 2, num_channels, 3, padding=1, bias=True),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.2, mode='fan_in')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # Shallow features
        feat = self.head(x)
        feat = feat + self.ms_head(feat)          # multi-scale + residual
        shallow = feat

        # Deep features — collect all RDB outputs for GFF
        rdb_outs = []
        for rdb in self.rdbs:
            feat = rdb(feat)
            rdb_outs.append(feat)

        # Global feature fusion
        feat = self.gff(torch.cat(rdb_outs, dim=1))
        feat = self.global_attn(feat)
        feat = feat + shallow                      # global residual

        # Multi-scale in deep features
        feat = feat + self.ms_deep(feat)

        # Reconstruct
        out = self.tail(feat)

        # Global skip connection (image-level residual)
        return out + x


# Quick sanity check
model = MSRDAN(num_features=64, num_rdb=8, growth_rate=32).to(DEVICE)
dummy = torch.randn(1, 3, 256, 256).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"Model output shape: {out.shape}")  # Should be [1, 3, 256, 256]
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Model output shape: torch.Size([1, 3, 256, 256])
Total parameters: 1,546,141


In [7]:
# CELL 7: Loss Functions
# ============================================================
class CharbonnierLoss(nn.Module):
    """Charbonnier loss — smoother than L1, more robust than L2."""
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        diff = pred - target
        return torch.mean(torch.sqrt(diff * diff + self.eps))


class FrequencyLoss(nn.Module):
    """L1 loss in the FFT frequency domain to preserve high-frequency detail."""
    def forward(self, pred, target):
        pred_fft   = torch.fft.rfft2(pred,   norm='ortho')
        target_fft = torch.fft.rfft2(target, norm='ortho')
        return F.l1_loss(torch.abs(pred_fft), torch.abs(target_fft))


class CombinedLoss(nn.Module):
    def __init__(self, w_charb=1.0, w_freq=0.1, w_mse=0.5):
        super().__init__()
        self.charb  = CharbonnierLoss()
        self.freq   = FrequencyLoss()
        self.mse    = nn.MSELoss()
        self.w_charb = w_charb
        self.w_freq  = w_freq
        self.w_mse   = w_mse

    def forward(self, pred, target):
        l_charb = self.charb(pred, target)
        l_freq  = self.freq(pred, target)
        l_mse   = self.mse(pred, target)
        return self.w_charb * l_charb + self.w_freq * l_freq + self.w_mse * l_mse

In [8]:
# CELL 8: PSNR Metric
# ============================================================
def compute_psnr(pred, target, max_val=1.0):
    """Compute Peak Signal-to-Noise Ratio (dB)."""
    mse = F.mse_loss(pred, target)
    if mse == 0:
        return float('inf')
    return 10 * math.log10(max_val ** 2 / mse.item())

In [9]:
# CELL 9: Training Config
# ============================================================
CONFIG = {
    "batch_size"      : 8,
    "num_epochs"      : 100,
    "lr"              : 2e-4,
    "min_lr"          : 1e-6,
    "weight_decay"    : 1e-4,
    "img_size"        : 256,
    "num_workers"     : 2,
    "save_dir"        : "/content/checkpoints",
    "save_every"      : 5,          # Save checkpoint every N epochs
    "warmup_epochs"   : 5,
    "grad_clip"       : 1.0,
}

os.makedirs(CONFIG["save_dir"], exist_ok=True)

In [10]:
# CELL 10: DataLoaders
# ============================================================
train_dataset = SRDataset(lr_train_paths, hr_train_paths, augment=True,  img_size=CONFIG["img_size"])
val_dataset   = SRDataset(lr_val_paths,   hr_val_paths,   augment=False, img_size=CONFIG["img_size"])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

print(f"Train batches: {len(train_loader)} | Val samples: {len(val_dataset)}")

Train batches: 376 | Val samples: 100


In [11]:
# CELL 11: Optimizer & Scheduler
# ============================================================
model = MSRDAN(num_features=64, num_rdb=8, growth_rate=32).to(DEVICE)

optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
    betas=(0.9, 0.999),
    weight_decay=CONFIG["weight_decay"],
)

# Cosine annealing with warm restarts
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["num_epochs"] - CONFIG["warmup_epochs"],
    eta_min=CONFIG["min_lr"],
)

criterion = CombinedLoss(w_charb=1.0, w_freq=0.1, w_mse=0.5).to(DEVICE)

In [ ]:
# CELL 12: Training Loop
# ============================================================
def warmup_lr(optimizer, epoch, warmup_epochs, base_lr):
    """Linear LR warmup."""
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs
        for pg in optimizer.param_groups:
            pg['lr'] = lr


def train_one_epoch(model, loader, optimizer, criterion, device, epoch, warmup_epochs, base_lr):
    model.train()
    warmup_lr(optimizer, epoch, warmup_epochs, base_lr)
    total_loss = 0.0
    total_psnr = 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} [Train]", leave=False)
    for lr_imgs, hr_imgs in pbar:
        lr_imgs = lr_imgs.to(device, non_blocking=True)
        hr_imgs = hr_imgs.to(device, non_blocking=True)

        preds = model(lr_imgs)
        loss  = criterion(preds, hr_imgs)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optimizer.step()

        with torch.no_grad():
            psnr = compute_psnr(preds.clamp(0, 1), hr_imgs)
        total_loss += loss.item()
        total_psnr += psnr
        pbar.set_postfix(loss=f"{loss.item():.4f}", psnr=f"{psnr:.2f}")

    n = len(loader)
    return total_loss / n, total_psnr / n


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_psnr = 0.0
    for lr_imgs, hr_imgs in loader:
        lr_imgs = lr_imgs.to(device, non_blocking=True)
        hr_imgs = hr_imgs.to(device, non_blocking=True)
        preds = model(lr_imgs).clamp(0, 1)
        total_psnr += compute_psnr(preds, hr_imgs)
    return total_psnr / len(loader)


best_psnr = 0.0
train_losses, train_psnrs, val_psnrs = [], [], []

print("=" * 60)
print("Starting Training")
print("=" * 60)

for epoch in range(CONFIG["num_epochs"]):
    t0 = time.time()

    tr_loss, tr_psnr = train_one_epoch(
        model, train_loader, optimizer, criterion,
        DEVICE, epoch, CONFIG["warmup_epochs"], CONFIG["lr"]
    )
    val_psnr = validate(model, val_loader, DEVICE)

    # Step scheduler after warmup
    if epoch >= CONFIG["warmup_epochs"]:
        scheduler.step()

    cur_lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0

    train_losses.append(tr_loss)
    train_psnrs.append(tr_psnr)
    val_psnrs.append(val_psnr)

    print(f"Epoch [{epoch+1:03d}/{CONFIG['num_epochs']}] "
          f"Loss: {tr_loss:.4f} | Train PSNR: {tr_psnr:.2f} dB | "
          f"Val PSNR: {val_psnr:.2f} dB | LR: {cur_lr:.2e} | {elapsed:.1f}s")

    # Save best
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_psnr'   : val_psnr,
        }, os.path.join(CONFIG["save_dir"], "best_model.pth"))
        print(f"  *** New best PSNR: {best_psnr:.2f} dB — model saved ***")

    # Periodic checkpoints
    if (epoch + 1) % CONFIG["save_every"] == 0:
        torch.save(model.state_dict(),
                   os.path.join(CONFIG["save_dir"], f"epoch_{epoch+1}.pth"))

print(f"\nTraining complete. Best Val PSNR: {best_psnr:.2f} dB")

Starting Training


Epoch [001/100] Loss: 0.1857 | Train PSNR: 20.13 dB | Val PSNR: 19.75 dB | LR: 4.00e-05 | 124.2s
  *** New best PSNR: 19.75 dB — model saved ***


Epoch [002/100] Loss: 0.0525 | Train PSNR: 23.23 dB | Val PSNR: 20.84 dB | LR: 8.00e-05 | 122.5s
  *** New best PSNR: 20.84 dB — model saved ***


Epoch [003/100] Loss: 0.0468 | Train PSNR: 23.95 dB | Val PSNR: 21.20 dB | LR: 1.20e-04 | 122.5s
  *** New best PSNR: 21.20 dB — model saved ***


Epoch [004/100] Loss: 0.0444 | Train PSNR: 24.24 dB | Val PSNR: 21.40 dB | LR: 1.60e-04 | 122.5s
  *** New best PSNR: 21.40 dB — model saved ***


...

Epoch [037/100] Loss: 0.0398 | Train PSNR: 24.88 dB | Val PSNR: 22.06 dB | LR: 1.49e-04 | 122.6s
  *** New best PSNR: 22.06 dB — model saved ***


Epoch [038/100] Loss: 0.0398 | Train PSNR: 24.88 dB | Val PSNR: 22.06 dB | LR: 1.46e-04 | 122.6s
  *** New best PSNR: 22.06 dB — model saved ***


Epoch [039/100] Loss: 0.0398 | Train PSNR: 24.86 dB | Val PSNR: 22.02 dB | LR: 1.43e-04 | 122.6s


Epoch [040/100] Loss: 0.0397 | Train PSNR: 24.91 dB | Val PSNR: 22.06 dB | LR: 1.40e-04 | 122.6s


Epoch 41 [Train]:  76%|███████▋  | 287/376 [01:30<00:28,  3.16it/s, loss=0.0412, psnr=24.82]

In [ ]:
# CELL 13: Load Best Model & Final Validation
# ============================================================
ckpt = torch.load(os.path.join(CONFIG["save_dir"], "best_model.pth"), map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded best model from epoch {ckpt['epoch']+1}, Val PSNR: {ckpt['val_psnr']:.2f} dB")

final_psnr = validate(model, val_loader, DEVICE)
print(f"Final Validation PSNR: {final_psnr:.2f} dB")

In [ ]:
# CELL 14: Export to ONNX
# ============================================================
model.eval()
dummy_input = torch.randn(1, 3, 256, 256).to(DEVICE)
onnx_path = "/content/msrdan_sr.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input":  {0: "batch_size"},
        "output": {0: "batch_size"},
    }
)
print(f"ONNX model exported to: {onnx_path}")

# Verify ONNX
try:
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model verification: PASSED")
except ImportError:
    print("onnx not installed — skipping verification (install with: pip install onnx)")
